# Stage 1
## Baseline Pipeline
The main goal is to take a video file and its transcript, and produce correctly-timed clips. I am investigating if the pipeline from

``video.mp4 + transcript.json`` ---> ``clips.mp4``

I apply the following structure to represent the baseline pipeline:
1. **Ingest** video and transcript files
2. **Locating segments**: Figure out which parts of the transcript are the ones we want to turn into clips. In our design, the transcript itself tells us this (each segment has a make_clip: true/false flag, like a producer would tag it).
3. **Determine timestamps**: The transcript has text but no timing, so we need something to figure out when in the audio each piece of text is actually spoken. That's the "alignment" step, and it's the one that normally calls Whisper or AssemblyAI.
4. **Cut the video**: Once we know start/end times for a clip-worthy segment, slice the source video there.
5. **Export the clips**: Write them out as MP4s, plus a manifest describing what we made.

My thinking is that if the locate --> align --> cut --> export chain isn't solid, none of the "smart" incremental logic in later phases would have nothing reliable to build on top of. Caching and diffing only makes sense once we can trust the thing being cached

In [6]:
!pip install -q whisperx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 775.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Phase 1: Ingest video and transcript files
We ingest the video and its transcript. The transcript has text only, and has no timestamps. Each segment is tagged `make_clip: true/false`, like a producer would mark up a transcript by hand.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import glob
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Projects/LEC AI Project")

matches = glob.glob(str(DRIVE_ROOT / "**" / "brewster_kahle_transcript.json"), recursive=True)
if not matches:
    raise FileNotFoundError(f"Couldn't find brewster_kahle_transcript.json under {DRIVE_ROOT}.")
TRANSCRIPT_PATH = Path(matches[0])

transcript = json.load(open(TRANSCRIPT_PATH))
segments = transcript["segments"]

print(f"Loaded transcript: {TRANSCRIPT_PATH}")
print(f"video_id: {transcript['video_id']}")
print(f"{len(segments)} segments total, {sum(s['make_clip'] for s in segments)} flagged make_clip=true\n")

for s in segments[:5]:
    flag = "CLIP" if s["make_clip"] else "    "
    print(f"[{flag}] {s['id']:8} {s['speaker']:10} {s['text'][:70]}")
print("...")

video_matches = glob.glob(str(DRIVE_ROOT / "**" / "*Brewster Kahle*Interview*.mp4"), recursive=True)
if not video_matches:
    raise FileNotFoundError(f"Couldn't find the Brewster Kahle interview video under {DRIVE_ROOT}.")
VIDEO_PATH = Path(video_matches[0])

# Scoped per-video folder for this run's downstream writes (state.json, decisions.log,
# clips), kept separate from wherever TRANSCRIPT_PATH happens to live, so this run's
# artifacts never mix with the AI Ethics run's.
VIDEO_DIR = DRIVE_ROOT / "brewster_kahle"
VIDEO_DIR.mkdir(exist_ok=True)

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", str(VIDEO_PATH)],
    capture_output=True, text=True, check=True,
)
video_duration = float(probe.stdout.strip())
print(f"\nLoaded video: {VIDEO_PATH.name} ({video_duration:.1f}s = {video_duration/60:.2f} min)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded transcript: /content/drive/MyDrive/Projects/LEC AI Project/brewster_kahle/brewster_kahle_transcript.json
video_id: brewster_kahle_interview
140 segments total, 0 flagged make_clip=true

[    ] seg_001  Brewster Kahle Record and post everywhere!
[    ] seg_002  Ian Milligan Perfect.
[    ] seg_003  Brewster Kahle And please do with public domain!
[    ] seg_004  Ian Milligan OK, great. Well, that makes it easy because I know Caitlin reviewed th
[    ] seg_005  Brewster Kahle Would love it, post it – public domain.
...

Loaded video: Brewster Kahle - Interview - 26 Feb 2021.mp4 (3613.3s = 60.22 min)


## Phase 2: Locating segments


In [2]:
matches = glob.glob(str(DRIVE_ROOT / "**" / "brewster_kahle_transcript_flagged.json"), recursive=True)
if not matches:
    raise FileNotFoundError(
        f"Couldn't find brewster_kahle_transcript_flagged.json under {DRIVE_ROOT}. "
        "Upload the edited transcript into your Drive folder first."
    )
FLAGGED_PATH = Path(matches[0])
flagged_transcript = json.load(open(FLAGGED_PATH))
flagged_segments = flagged_transcript["segments"]

print(f"Loaded edited transcript: {FLAGGED_PATH}\n")

clip_segments = [s for s in flagged_segments if s.get("make_clip")]

if not clip_segments:
    print(f"0 of {len(flagged_segments)} segments are flagged make_clip=true.")
else:
    for s in clip_segments:
        title = s.get("clip_title", "(no clip_title set)")
        print(f"{s['id']:8} \"{title}\"")
        print(f"         {s['text']}\n")

Loaded edited transcript: /content/drive/MyDrive/Projects/LEC AI Project/brewster_kahle/brewster_kahle_transcript_flagged.json

seg_008  "The Question That Started It All"
         So Brewster, I thought we might start if we could go way back in time to your time at MIT and, you know, where the origin of your idea of creating a new "Library of Alexandria" came from?

seg_010  "Plan B: The Library of Alexandria, Version Two"
         He said "Brewster, you're an idealist." I said, "yes and a technologist, yes." And he said, "paint a portrait that is better because of the future. That's better because of your technology." And this turned out to be extremely hard! So I walked back and forth over the bridge, back and forth from Boston to Cambridge as I walked back and forth to school. And I could only come up with two ideas. And one was too obvious. So I thought other people would do it. So I went for the other one, which was to try to protect people's privacy because people are just going

## Phase 3: Determining the timestamps
This is "forced alignment". Given text and audio, we find the start/end time of every word.

1. A sentence has no timestamp of its own, because the audio is just a continuous stream of sound, with no marker for where one sentence starts or ends.
2. So instead of trying to locate whole sentences directly, we locate the individual words that make them up, since a word is short and distinctive enough for an alignment model to pin down precisely.
3. Once every word in a segment has a start/end time, the segment's own start is just its first word's start, and its end is its last word's end.
4. We use Whisper-style forced alignment (via whisperx) to snap our own known transcript text onto the audio, rather than trusting a blind transcription. In that manner, I think the timestamps would be tied to text we already know is correct

In [3]:
def estimate_rough_boundaries(segments, total_duration):
    """
    Rough, proportional starting guess for each segment's start/end, based on
    how many words it has relative to the whole transcript. This is NOT the
    final timestamp -- it's just a coarse window for whisperx's CTC aligner
    to search within, refined into precise word-level timing in the next step.
    """
    word_counts = [len(s["text"].split()) for s in segments]
    total_words = sum(word_counts)

    rough = []
    cursor = 0.0
    for seg, wc in zip(segments, word_counts):
        frac = wc / total_words
        duration = frac * total_duration
        rough.append({
            "id": seg["id"],
            "text": seg["text"],
            "start": round(cursor, 2),
            "end": round(cursor + duration, 2),
        })
        cursor += duration
    return rough


LEAD_IN_SEC = 25

rough_boundaries = estimate_rough_boundaries(flagged_segments, video_duration - LEAD_IN_SEC)
for r in rough_boundaries:
    r["start"] += LEAD_IN_SEC
    r["end"] += LEAD_IN_SEC

print(f"Shifted rough boundaries by {LEAD_IN_SEC}s. First 5:\n")
for r in rough_boundaries[:5]:
    print(f"{r['id']:8} [{r['start']:6.1f}s - {r['end']:6.1f}s]  {r['text'][:60]}")

Shifted rough boundaries by 25s. First 5:

seg_001  [  25.0s -   26.6s]  Record and post everywhere!
seg_002  [  26.6s -   27.0s]  Perfect.
seg_003  [  27.0s -   29.4s]  And please do with public domain!
seg_004  [  29.4s -   49.6s]  OK, great. Well, that makes it easy because I know Caitlin r
seg_005  [  49.6s -   52.8s]  Would love it, post it – public domain.


In the cell below, we install whisperx, load its wav2vec2-based CTC alignment model, and refines our rough per-segment guesses into precise word-level timestamps

In [ ]:
import numpy
print(numpy.__version__)

In [ ]:
!pip install -q --force-reinstall torch torchvision torchaudio

In [9]:
!pip uninstall -y numpy
!pip install "numpy<2.1"

Found existing installation: numpy 2.5.2
Uninstalling numpy-2.5.2:
  Successfully uninstalled numpy-2.5.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 90.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.6 requires numpy>=2.1.0, but you have numpy 2.0.2 which is incompatible.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 2.0.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which i

In [4]:
import whisperx
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Extract audio from the video once, since whisperx aligns against an audio
# waveform, not the mp4 container directly.
AUDIO_PATH = VIDEO_DIR / "interview_audio.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", str(VIDEO_PATH), "-ac", "1", "-ar", "16000", str(AUDIO_PATH)],
    check=True, capture_output=True,
)
print(f"Extracted audio: {AUDIO_PATH}")

Using device: cuda
Extracted audio: /content/drive/MyDrive/Projects/LEC AI Project/brewster_kahle/interview_audio.wav


In [5]:
import torch
import whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

asr_model = whisperx.load_model("small", device, compute_type="int8" if device == "cpu" else "float16")
asr_result = asr_model.transcribe(str(AUDIO_PATH), batch_size=16)

print(f"ASR produced {len(asr_result['segments'])} segments, language={asr_result['language']}")
for seg in asr_result["segments"][:5]:
    print(f"[{seg['start']:6.1f}s - {seg['end']:6.1f}s] {seg['text']}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

2026-08-16 13:32:08 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-08-16 13:32:08 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

2026-08-16 13:32:18 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
ASR produced 145 segments, language=en
[   0.0s -   19.2s]  Recording. I'll change my, I think I have, let's see, how do I do this? Oh, now I'm in this mode. Gallery, good. All right, then. Excellent.
[  20.3s -   44.4s]  Excellent. So just to start. Record and post everywhere. Perfect. And please do public domain. OK, great. Well, that makes it easy because I know I know Caitlin reviewed the form. But the three checkmarks on the form are mostly just making sure you're comfortable with recording interview, transcribing the interview, and then having any quotations in your interview being ascribed by name to you.
[  46.0s -   71.3s]  would love it, post it, public domain. Perfect. Wonderful. Yeah. And then hopefully we can put it on the internet archive. So I will, that would be great. Excellent. Well, I thought, I thought we'd just jump in because hopefully we can cover a fair bit of ground t

In [22]:
import re

align_model, align_metadata = whisperx.load_align_model(language_code="en", device=device)
audio = whisperx.load_audio(str(AUDIO_PATH))

aligned_asr = whisperx.align(
    asr_result["segments"], align_model, align_metadata, audio, device,
    return_char_alignments=False,
)

asr_words = []
for seg in aligned_asr["segments"]:
    for w in seg.get("words", []):
        if w.get("start") is not None and w.get("end") is not None:
            asr_words.append({
                "word": re.sub(r"[^a-z0-9]", "", w["word"].lower()),
                "start": w["start"],
                "end": w["end"],
            })

print(f"Got {len(asr_words)} real, audio-aligned ASR words")

Got 8801 real, audio-aligned ASR words


In [27]:
from difflib import SequenceMatcher
import re

def normalize(t):
    return re.sub(r"[^a-z0-9 ]", "", t.lower()).split()

target_words_flat = []
segment_ranges = []
for seg in flagged_segments:
    words = normalize(seg["text"])
    start_idx = len(target_words_flat)
    target_words_flat.extend(words)
    segment_ranges.append((seg, start_idx, len(target_words_flat)))

asr_word_list = [w["word"] for w in asr_words]

sm = SequenceMatcher(None, target_words_flat, asr_word_list, autojunk=False)
blocks = [b for b in sm.get_matching_blocks() if b.size > 0]

print(f"Global diff: {len(blocks)} matching blocks, "
      f"{sum(b.size for b in blocks)} of {len(target_words_flat)} words matched overall")

new_boundaries = []
n_weak = 0
last_good_end = 0.0

for seg, start_idx, end_idx in segment_ranges:
    overlapping = [b for b in blocks if b.a < end_idx and b.a + b.size > start_idx]

    if overlapping:
        asr_indices = []
        for b in overlapping:
            lo = max(b.a, start_idx) - b.a + b.b
            hi = min(b.a + b.size, end_idx) - b.a + b.b
            asr_indices.extend(range(lo, hi))
        start = asr_words[min(asr_indices)]["start"]
        end = asr_words[max(asr_indices)]["end"]
        last_good_end = end
    else:
        start = last_good_end
        end = start + 1.5
        n_weak += 1
        print(f"WEAK MATCH: {seg['id']} -- {seg['text'][:50]}")

    new_boundaries.append({"id": seg["id"], "start": start, "end": end, "text": seg["text"]})

print(f"\nMatched {len(new_boundaries) - n_weak} of {len(new_boundaries)} segments confidently "
      f"({n_weak} weak). First 5:")
for r in new_boundaries[:5]:
    print(f"{r['id']:8} [{r['start']:6.1f}s - {r['end']:6.1f}s]  {r['text'][:60]}")

Global diff: 640 matching blocks, 8221 of 9035 words matched overall
WEAK MATCH: seg_021 -- OK
WEAK MATCH: seg_031 -- Yeah.
WEAK MATCH: seg_033 -- Yeah.
WEAK MATCH: seg_036 -- Yes.
WEAK MATCH: seg_038 -- Yeah, yeah.
WEAK MATCH: seg_045 -- Yeah.
WEAK MATCH: seg_070 -- Yes.
WEAK MATCH: seg_074 -- OK.
WEAK MATCH: seg_082 -- Yeah.
WEAK MATCH: seg_092 -- Yeah.
WEAK MATCH: seg_094 -- Yeah.
WEAK MATCH: seg_098 -- OK.
WEAK MATCH: seg_102 -- Yeah.
WEAK MATCH: seg_104 -- Yeah.
WEAK MATCH: seg_110 -- Yeah.
WEAK MATCH: seg_112 -- Yeah, which...
WEAK MATCH: seg_114 -- Yeah.

Matched 123 of 140 segments confidently (17 weak). First 5:
seg_001  [  23.8s -   25.3s]  Record and post everywhere!
seg_002  [  25.7s -   26.0s]  Perfect.
seg_003  [  26.7s -   28.2s]  And please do with public domain!
seg_004  [  28.7s -   44.3s]  OK, great. Well, that makes it easy because I know Caitlin r
seg_005  [  46.1s -   48.6s]  Would love it, post it – public domain.


In [28]:
# Load the CTC alignment model for English. This downloads from Hugging Face
# the first time it runs -- fine here since Colab has real internet access.
align_model, align_metadata = whisperx.load_align_model(language_code="en", device=device)

# whisperx expects segments shaped like Whisper's own output: a list of dicts
# with "start", "end", "text" -- which is exactly what rough_boundaries is.
audio = whisperx.load_audio(str(AUDIO_PATH))
aligned_result = whisperx.align(
    new_boundaries, align_model, align_metadata, audio, device,
    return_char_alignments=False,
)

print(f"\nAligned {len(aligned_result['segments'])} segments. First 5 refined boundaries:\n")
for seg in aligned_result["segments"][:5]:
    print(f"{seg.get('id', '?'):8} [{seg['start']:6.2f}s - {seg['end']:6.2f}s]  {seg['text'][:60]}")

2026-08-16 14:15:29 - whisperx.alignment - WARNING - Failed to align segment ("And this is at Thinking Machines, then?"): backtrack failed, resorting to original
2026-08-16 14:16:07 - whisperx.alignment - WARNING - Failed to align segment ("By Jeff Rothenberg?"): backtrack failed, resorting to original

Aligned 691 segments. First 5 refined boundaries:

?        [ 23.83s -  25.29s]  Record and post everywhere!
?        [ 25.65s -  25.95s]  Perfect.
?        [ 26.67s -  28.20s]  And please do with public domain!
?        [ 28.70s -  29.32s]  OK, great.
?        [ 30.14s -  44.31s]  Well, that makes it easy because I know Caitlin reviewed the


In [29]:
# Flatten every aligned word, across all of whisperx's (re-split) segments,
# into one ordered timeline -- word order survives the resegmentation even
# though sentence grouping didn't.
all_words = []
for seg in aligned_result["segments"]:
    for w in seg.get("words", []):
        all_words.append(w)  # each has "word", and usually "start"/"end"/"score"

print(f"Total aligned words: {len(all_words)}")

# Sanity check before trusting positional re-attribution: our own transcript's
# word count (via .split()) must match whisperx's word count exactly, or the
# word_cursor below will silently drift and misattribute timings past
# whichever segment first goes out of sync.
our_word_count = sum(len(seg["text"].split()) for seg in flagged_segments)
print(f"Our transcript's word count (via .split()): {our_word_count}")
if our_word_count != len(all_words):
    print(f"WARNING: mismatch of {abs(our_word_count - len(all_words))} words -- "
          f"re-attribution below will drift for segments after the first divergence. "
          f"Likely cause: bracketed non-verbal annotations ([Laughs], [pause], etc.) "
          f"being tokenized differently by whisperx than by .split().\n")
else:
    print("Word counts match exactly -- positional re-attribution is safe.\n")

# Re-attribute words back onto OUR original 140 segments, in order, by
# consuming len(segment.split()) words per segment -- same idea as the
# PocketSphinx _attribute_words step, just fed by whisperx's word timeline.
word_cursor = 0
resolved = []
for seg in flagged_segments:
    n_words = len(seg["text"].split())
    seg_words = all_words[word_cursor: word_cursor + n_words]
    word_cursor += n_words

    timed = [w for w in seg_words if w.get("start") is not None and w.get("end") is not None]
    if timed:
        start = min(w["start"] for w in timed)
        end = max(w["end"] for w in timed)
        confidence = "aligned" if len(timed) == len(seg_words) else "interpolated"
    else:
        start = end = None
        confidence = "unresolved"

    resolved.append({
        "id": seg["id"], "text": seg["text"], "speaker": seg.get("speaker"),
        "make_clip": seg.get("make_clip", False), "clip_title": seg.get("clip_title"),
        "start": start, "end": end, "confidence": confidence,
    })

n_unresolved = sum(1 for r in resolved if r["confidence"] == "unresolved")
print(f"Resolved {len(resolved)} segments ({n_unresolved} unresolved)\n")

for r in resolved[:5]:
    s = f"{r['start']:.2f}s" if r["start"] is not None else "?"
    e = f"{r['end']:.2f}s" if r["end"] is not None else "?"
    print(f"{r['id']:8} [{s:>8} - {e:>8}] ({r['confidence']:12}) {r['text'][:55]}")

Total aligned words: 9033
Our transcript's word count (via .split()): 9043

Resolved 140 segments (2 unresolved)

seg_001  [  23.83s -   25.29s] (aligned     ) Record and post everywhere!
seg_002  [  25.65s -   25.95s] (aligned     ) Perfect.
seg_003  [  26.67s -   28.20s] (aligned     ) And please do with public domain!
seg_004  [  28.70s -   44.31s] (aligned     ) OK, great. Well, that makes it easy because I know Cait
seg_005  [  46.10s -   48.56s] (aligned     ) Would love it, post it – public domain.


In the next cell, we perform a spot-check a few points spread across the full 12 minutes to test whether alignment held up over time, since early segments are the easiest case.

This cell cuts short audio snippets around a few chosen timestamps and plays them inline, so I can listen and compare against the expected text

In [30]:
from IPython.display import Audio, display

CHECK_IDS = ["seg_005", "seg_057", "seg_108", "seg_071"]

resolved_by_id = {r["id"]: r for r in resolved}

for check_id in CHECK_IDS:
    r = resolved_by_id[check_id]
    start, end = r["start"], r["end"]
    pad = 0.3
    clip_start = max(0, start - pad)
    clip_duration = (end + pad) - clip_start
    snippet_path = VIDEO_DIR / f"_check_{check_id}.wav"

    subprocess.run(
        ["ffmpeg", "-y", "-ss", str(clip_start), "-i", str(AUDIO_PATH),
         "-t", str(clip_duration), str(snippet_path)],
        check=True, capture_output=True,
    )

    print(f"\n{check_id}  [{start:.2f}s - {end:.2f}s]  duration={clip_duration:.2f}s  confidence={r['confidence']}")
    print(f"Expected text: \"{r['text']}\"")
    display(Audio(str(snippet_path)))


seg_005  [46.10s - 48.56s]  duration=3.06s  confidence=aligned
Expected text: "Would love it, post it – public domain."



seg_057  [1660.93s - 1661.75s]  duration=1.42s  confidence=aligned
Expected text: "Yeah, that's great."



seg_108  [2900.92s - 2901.28s]  duration=0.96s  confidence=aligned
Expected text: "Yes."



seg_071  [1967.43s - 1996.24s]  duration=29.42s  confidence=aligned
Expected text: "[Laughs] And we launched it at the Bancroft Library with Larry Lessig and the head of the journalism school and the head of the Bancroft Library. You know, the walls and the ancient things. I mean, the Bancroft Library, University of California, Berkeley is a beautiful environment, right? And Berkeley knew exactly what they were doing by going and giving this new upstart idea that kind of a venue for launching the Wayback Machine."


## Phase 4: Cutting the video

In this section we follow the following sequence:
1. We tell ffmpeg to open the source video
2. Jump to a start time
3. Keep everything until an end time
4. Discard the rest
5. Save that stretch as its own file

The following details are crucial in realising the result:

1. **Padding**: Our resolved timestamps mark exactly where the aligned words begin and end — but "exactly" here is a little too exact. If a clip starts at the literal first millisecond of the first word, it can feel abrupt, like you walked in mid-sentence even though you didn't. So we pad slightly: start a touch earlier (say 0.15s) and end a touch later (say 0.35s, a bit more generous on the tail so trailing consonants don't get chopped). This provides a cosmetic breathing room, not a correction to the alignment itself.

2. **Frame-accurate seeking vs. fast seeking**: ffmpeg has two ways to jump to a timestamp: seek before decoding starts (fast, but can only land on the nearest keyframe, which might be a second or more away from where you actually asked), or seek after opening the video and decode forward to the exact frame (slower, but precise to the frame). For a 12-minute source video, speed doesn't matter much, but precision does — landing a second early or late on a short clip is very noticeable. So we use the slower, accurate method (-i before -ss, not after) and re-encode the output rather than doing a lossless stream-copy, since stream-copying can only cut on keyframe boundaries too.

### Output:
Each flagged segment becomes its own file (clip_001.mp4, clip_002.mp4, ...), and we also build a small manifest recording which original segment each clip came from, its title, its timing, and its confidence level — so nothing about how a clip was chosen or trusted gets lost once it's just a video file sitting in a folder.

In [31]:
CLIPS_DIR = VIDEO_DIR / "clips"
CLIPS_DIR.mkdir(exist_ok=True)

CLIP_PAD_START = 0.15
CLIP_PAD_END = 0.35

def cut_clip(source_video, start, end, out_path):
    padded_start = max(0.0, start - CLIP_PAD_START)
    padded_end = end + CLIP_PAD_END
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(source_video),
         "-ss", str(padded_start), "-to", str(padded_end),
         "-c:v", "libx264", "-c:a", "aac", "-avoid_negative_ts", "make_zero",
         str(out_path)],
        check=True, capture_output=True,
    )

clip_records = []
clip_index = 0
for r in resolved:
    if not r["make_clip"]:
        continue
    clip_index += 1
    filename = f"clip_{clip_index:03d}.mp4"
    out_path = CLIPS_DIR / filename
    cut_clip(VIDEO_PATH, r["start"], r["end"], out_path)
    clip_records.append({
        "clip_id": filename.replace(".mp4", ""),
        "filename": filename,
        "segment_id": r["id"],
        "clip_title": r["clip_title"],
        "start": round(r["start"], 3),
        "end": round(r["end"], 3),
        "duration": round(r["end"] - r["start"], 3),
        "confidence": r["confidence"],
        "text": r["text"],
    })
    print(f"Cut {filename}  ({clip_records[-1]['duration']:.2f}s)  \"{r['clip_title']}\"")

print(f"\nCut {len(clip_records)} clips into {CLIPS_DIR}")

Cut clip_001.mp4  (10.61s)  "The Question That Started It All"
Cut clip_002.mp4  (75.21s)  "Plan B: The Library of Alexandria, Version Two"
Cut clip_003.mp4  (89.58s)  "Before There Was a Web"
Cut clip_004.mp4  (45.52s)  "Two Companies, One Contract"
Cut clip_005.mp4  (29.34s)  "Rain on You Like Frogs"
Cut clip_006.mp4  (11.66s)  "Either Way, You're One of Us"
Cut clip_007.mp4  (23.78s)  "Don't Do It Too Soon"
Cut clip_008.mp4  (63.35s)  "October 24, 2001"
Cut clip_009.mp4  (28.82s)  "Launching at the Bancroft Library"
Cut clip_010.mp4  (74.66s)  "The Only Place the Word Copyright Appeared"
Cut clip_011.mp4  (59.83s)  "The Stupidest Thing Since Hiroshima"
Cut clip_012.mp4  (37.57s)  "Taking Copyright to the Supreme Court"
Cut clip_013.mp4  (49.81s)  "The Bookmobile, Built in Two Weeks"
Cut clip_014.mp4  (25.24s)  "Free to the People"
Cut clip_015.mp4  (86.44s)  "A Home for the Commons"
Cut clip_016.mp4  (39.95s)  "Teaching the New Overlord"

Cut 16 clips into /content/drive/MyDrive/Pro

## Phase 5: Exporting the clips

Based on our original 5-step structure, we avoid leaving 21 loose mp4 files sitting in a folder.

1. We produce a manifest that ties each clip back to its source segment, title, timing, and confidence (the clip_records list we already built in the cutting cell has everything needed)
2. This helps so that, say, six months from now, when someone asks "where did clip_011 come from and how sure are we about its timing," I can provide an answer without having to re-run the whole pipeline.

In [34]:
import hashlib
from datetime import datetime, timezone

def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

MANIFEST_PATH = VIDEO_DIR / "manifest.json"
STATE_PATH = VIDEO_DIR / "state.json"

# --- manifest.json: what clips exist and where they came from ---
manifest = {
    "video_id": transcript["video_id"],
    "source_video": str(VIDEO_PATH),
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "clips": clip_records,
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

# --- state.json: per-segment text fingerprints, for future incremental runs ---
segment_state = {
    r["id"]: {
        "text_hash": text_hash(r["text"]),
        "start": round(r["start"], 3) if r["start"] is not None else None,
        "end": round(r["end"], 3) if r["end"] is not None else None,
        "confidence": r["confidence"],
        "make_clip": r["make_clip"],
    }
    for r in resolved
}
full_transcript_hash = text_hash("".join(r["text"] for r in resolved))

run_history = []
if STATE_PATH.exists():
    run_history = json.load(open(STATE_PATH)).get("run_history", [])
run_history.append({
    "run_at": datetime.now(timezone.utc).isoformat(),
    "n_segments": len(resolved),
    "n_clips": len(clip_records),
    "transcript_hash": full_transcript_hash,
})

state = {
    "video_id": transcript["video_id"],
    "transcript_hash": full_transcript_hash,
    "segments": segment_state,
    "run_history": run_history,
}
with open(STATE_PATH, "w") as f:
    json.dump(state, f, indent=2)

print(f"Wrote {MANIFEST_PATH} ({len(clip_records)} clips)")
print(f"Wrote {STATE_PATH} ({len(segment_state)} segment hashes)")
print(f"Transcript hash: {full_transcript_hash}")

Wrote /content/drive/MyDrive/Projects/LEC AI Project/brewster_kahle/manifest.json (16 clips)
Wrote /content/drive/MyDrive/Projects/LEC AI Project/brewster_kahle/state.json (140 segment hashes)
Transcript hash: a10cd25f2eeaf77c


##Summary

Every time the pipeline is ran, it treats the transcript as brand new. It realigns every one of the 149 segments from scratch, even if we only fixed a typo in one sentence, lets say. Diffing is the step that looks at "the transcript we processed last time" versus "the transcript we're given now" and answers, segment by segment: did this change, and if so, how?

To solve this problem, we need to address the following, per segment:
1. Whether each text remained unchanged
2. Whether there were slight edits. For example, the text recognizably the same underlying sentence, but wording changed
3. Whether there was a major insertion, for example, new content that didn't exist in the old transcript at all
4. Or whether there was a section completely deleted: a section which existed, but was completely gone

Python's standard library has difflib.SequenceMatcher, which allows us to do exactly this kind of alignment. Its going to take two lists of segment texts,find the longest matching runs between them and reports the gaps as insert/delete/replace operations. In essence, we'll feed it the old segment list and the new segment list, and it'll tell us which stretches matched exactly, which were replaced, which were purely additions, and which were purely removals.

#Stage 2
In this stage, we begin by creating a 'new' test transcript, to enable us compare with the old transcript_ai_ethics_flagged.json file


The edits to be made to the new transcript entails reworded segment(s), inserted segment(s) and deleted segment(s). It'd be built as the edited copy alongside the original so we have a real old-vs-new pair to test the diff logic against it.

## Phase 1
In this section we give the agent its initial reasoning capability. Instead of blindly reprocessing everything, we let it compare the incoming transcript (second run) against what was recorded last time (state.json) and classify every segment as unchanged, edited, inserted, or deleted. This phase only produces that classification; deciding what to actually do about each case

In [1]:
from google.colab import files
uploaded = files.upload()  # pick brewster_kahle_transcript_flagged_v2.json from the file picker

Saving brewster_kahle_transcript_flagged_v2.json to brewster_kahle_transcript_flagged_v2.json


In [2]:
NEW_TRANSCRIPT_PATH = Path("brewster_kahle_transcript_flagged_v2.json")
new_transcript = json.load(open(NEW_TRANSCRIPT_PATH))
new_segments = new_transcript["segments"]

print(f"\nOld: {len(old_segments)} segments   New: {len(new_segments)} segments\n")

NameError: name 'Path' is not defined

In [39]:
old_matches = glob.glob(str(DRIVE_ROOT / "**" / "brewster_kahle_transcript_flagged.json"), recursive=True)
if not old_matches:
    raise FileNotFoundError("Couldn't find brewster_kahle_transcript_flagged.json to diff against.")
OLD_TRANSCRIPT_PATH = Path(old_matches[0])
old_transcript = json.load(open(OLD_TRANSCRIPT_PATH))
old_segments = old_transcript["segments"]

if not STATE_PATH.exists():
    raise FileNotFoundError(f"No state.json at {STATE_PATH} -- run Stage 1 / phase 5 (export) at least once first.")
old_state = json.load(open(STATE_PATH))

mismatches = [
    s["id"] for s in old_segments
    if old_state["segments"].get(s["id"], {}).get("text_hash") != text_hash(s["text"])
]
if mismatches:
    print(f"WARNING: {len(mismatches)} segment(s) don't match state.json's stored hash: {mismatches}")
else:
    print(f"Sanity check passed: state.json's {len(old_state['segments'])} stored hashes "
          f"match the transcript file it was built from.")

new_matches = glob.glob(str(DRIVE_ROOT / "**" / "brewster_kahle_transcript_flagged_v2.json"), recursive=True)
if not new_matches:
    raise FileNotFoundError("Couldn't find brewster_kahle_transcript_flagged_v2.json -- upload it to the Drive folder first.")
NEW_TRANSCRIPT_PATH = Path(new_matches[0])
new_transcript = json.load(open(NEW_TRANSCRIPT_PATH))
new_segments = new_transcript["segments"]

print(f"\nOld: {len(old_segments)} segments   New: {len(new_segments)} segments\n")

Sanity check passed: state.json's 140 stored hashes match the transcript file it was built from.


FileNotFoundError: Couldn't find brewster_kahle_transcript_flagged_v2.json -- upload it to the Drive folder first.

In [ ]:
old_ids = [s["id"] for s in old_segments]
new_ids = [s["id"] for s in new_segments]
old_hashes = [text_hash(s["text"]) for s in old_segments]
new_hashes = [text_hash(s["text"]) for s in new_segments]

# Match on TEXT CONTENT (via hash), not position -- this correctly handles
# insertions/deletions shifting everything after them, since SequenceMatcher
# finds the longest matching runs by value, not by index.
matcher = difflib.SequenceMatcher(a=old_hashes, b=new_hashes, autojunk=False)

classifications = []
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        for oi, ni in zip(range(i1, i2), range(j1, j2)):
            classifications.append({"kind": "unchanged", "old_id": old_ids[oi], "new_id": new_ids[ni],
                                     "old_text": old_segments[oi]["text"], "new_text": new_segments[ni]["text"]})
    elif tag == "replace":
        old_block, new_block = list(range(i1, i2)), list(range(j1, j2))
        # Same-length block -> pair positionally as edits (rewordings).
        # Any length mismatch left over -> pure delete/insert. A smarter
        # sub-match on similarity is exactly what embeddings will add next.
        for oi, ni in zip(old_block, new_block):
            classifications.append({"kind": "edited", "old_id": old_ids[oi], "new_id": new_ids[ni],
                                     "old_text": old_segments[oi]["text"], "new_text": new_segments[ni]["text"]})
        for oi in old_block[len(new_block):]:
            classifications.append({"kind": "deleted", "old_id": old_ids[oi], "new_id": None,
                                     "old_text": old_segments[oi]["text"], "new_text": None})
        for ni in new_block[len(old_block):]:
            classifications.append({"kind": "inserted", "old_id": None, "new_id": new_ids[ni],
                                     "old_text": None, "new_text": new_segments[ni]["text"]})
    elif tag == "delete":
        for oi in range(i1, i2):
            classifications.append({"kind": "deleted", "old_id": old_ids[oi], "new_id": None,
                                     "old_text": old_segments[oi]["text"], "new_text": None})
    elif tag == "insert":
        for ni in range(j1, j2):
            classifications.append({"kind": "inserted", "old_id": None, "new_id": new_ids[ni],
                                     "old_text": None, "new_text": new_segments[ni]["text"]})

summary = {}
for c in classifications:
    summary[c["kind"]] = summary.get(c["kind"], 0) + 1
print("Diff summary:", summary, "\n")

for c in classifications:
    if c["kind"] == "unchanged":
        continue
    if c["kind"] == "edited":
        print(f"[EDITED]   {c['old_id']} -> {c['new_id']}\n   old: {c['old_text'][:80]}\n   new: {c['new_text'][:80]}\n")
    elif c["kind"] == "inserted":
        print(f"[INSERTED] {c['new_id']}\n   new: {c['new_text'][:80]}\n")
    elif c["kind"] == "deleted":
        print(f"[DELETED]  {c['old_id']}\n   old: {c['old_text'][:80]}\n")

## Phase 2: Embedding transcript sentences
In this phase, we convert each segment's text into a vector (a list of numbers) that captures its meaning. We use a small pretrained model (all-MiniLM-L6-v2, via sentence-transformers) to do so. We'll compute one embedding per segment, cache it by that segment's text hash (so an unchanged segment's embedding is never recomputed on a future run), and use similarity scores to validate/refine the "edited" pairs diffing already found.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Small, fast, good general-purpose model -- not the biggest available, but
# more than enough resolution for telling "reworded" from "different idea."
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# The embedding cache: keyed by the SAME text_hash already used in state.json,
# so a future run can check "do I already have this one?" before calling the
# model at all. For now we just build it fresh for both old and new segments.
embedding_cache = {}

def get_embedding(text):
    h = text_hash(text)
    if h not in embedding_cache:
        embedding_cache[h] = embed_model.encode(text, normalize_embeddings=True)
    return embedding_cache[h]

def cosine_similarity(a, b):
    return float(np.dot(a, b))  # safe shortcut since embeddings are normalized above

# Embed every segment on both sides. Cache hits (identical text on old vs
# new) mean this genuinely does less work than embedding both sides blind.
for s in old_segments + new_segments:
    get_embedding(s["text"])

print(f"Computed {len(embedding_cache)} unique embeddings "
      f"(from {len(old_segments)} old + {len(new_segments)} new = "
      f"{len(old_segments) + len(new_segments)} total segments -- "
      f"the gap is cache hits from unchanged text)")

# Sanity check 1: our one known "edited" pair should score HIGH -- same idea, reworded.
old_024 = next(s for s in old_segments if s["id"] == "seg_024")["text"]
new_024 = next(s for s in new_segments if s["id"] == "seg_024")["text"]
sim_edited = cosine_similarity(get_embedding(old_024), get_embedding(new_024))
print(f"\nseg_024 old vs new similarity: {sim_edited:.3f}  (expect high -- same idea, reworded)")

# Sanity check 2: two genuinely unrelated segments should score LOW.
unrelated_a = old_segments[0]["text"]   # "to run, and then we kind of edit. daytime."
unrelated_b = next(s for s in old_segments if s["id"] == "seg_057")["text"]  # GDPR/ACM/BCS segment
sim_unrelated = cosine_similarity(get_embedding(unrelated_a), get_embedding(unrelated_b))
print(f"Unrelated pair similarity: {sim_unrelated:.3f}  (expect much lower)")

### Results
1. `seg_024`'s pair scored 0.948, which is very close to 1. That's the model correctly recognizing that the old and new wording of that segment ("StarFO students" garble vs. "staff and students") are expressing the same underlying statement, just with a cleaned up typo. This outlines a clear signature for a case that should be classified as "reuse the timing, this isn't a new idea" — high similarity, not a full rewrite.

2. The unrelated pair scored 0.086 which is barely above zero. That's the model correctly recognizing that "to run, and then we kind of edit. daytime." and the GDPR/ACM/BCS regulatory-alignment segment have nothing meaningfully in common. This is the other end of the scale: something a future decision engine should treat as "not the same idea at all."

## Phase 3: Alignment reuse
In this section, we outline 3 steps to address the question: What are this new transcript's correct timestamps, reusing whatever's still valid?

1. Step 1 is the decision cell, which answers "what should happen to each segment?"
2. Step 2 is the windowed patch-alignment + delta-propagation logic.It executes those decisions (i.e., do the small realignment, compute the shift, apply it)
3. Step 3 is writing `decisions.log`. It simply records the whole process

In [ ]:
SIMILARITY_THRESHOLD = 0.85  # tunable: above this, treat an edit as a minor reword

decisions = []
downstream_of_change = False  # flips true the first time we cross any change point

for c in classifications:
    if c["kind"] == "unchanged":
        if not downstream_of_change:
            decisions.append({
                "id": c["new_id"], "action": "REUSE",
                "reason": "unchanged text, no upstream edits before it -- old timestamp is trustworthy as-is",
            })
        else:
            decisions.append({
                "id": c["new_id"], "action": "VERIFY_SHIFT",
                "reason": "text unchanged, but occurs after an earlier edit/insertion/deletion -- "
                          "absolute timing may have shifted, needs offset re-verification",
            })

    elif c["kind"] == "edited":
        sim = cosine_similarity(get_embedding(c["old_text"]), get_embedding(c["new_text"]))
        if sim >= SIMILARITY_THRESHOLD:
            decisions.append({
                "id": c["new_id"], "action": "PATCH",
                "reason": f"minor reword (similarity {sim:.3f} >= {SIMILARITY_THRESHOLD}) -- "
                          f"realign this segment locally, not the whole file",
            })
        else:
            decisions.append({
                "id": c["new_id"], "action": "FULL_REALIGN",
                "reason": f"substantial rewrite (similarity {sim:.3f} < {SIMILARITY_THRESHOLD}) -- "
                          f"too different to trust a local patch",
            })
        downstream_of_change = True

    elif c["kind"] == "inserted":
        decisions.append({
            "id": c["new_id"], "action": "NEW",
            "reason": "no prior alignment exists for this segment -- must align fresh",
        })
        downstream_of_change = True

    elif c["kind"] == "deleted":
        decisions.append({
            "id": c["old_id"], "action": "REMOVE",
            "reason": "segment no longer exists in the new transcript",
        })
        downstream_of_change = True

summary = {}
for d in decisions:
    summary[d["action"]] = summary.get(d["action"], 0) + 1
print("Decision summary:", summary, "\n")

for d in decisions:
    if d["action"] == "REUSE":
        continue  # too many to print, least interesting
    print(f"Decision: {d['action']:<12} {d['id']}")
    print(f"Reason:   {d['reason']}\n")

In [ ]:
# Grounded estimate for mocking duration changes, since we have no real new
# audio for this synthetic v2 transcript yet -- swap for a real windowed
# whisperx call once actual audio exists.
total_old_duration = sum(v["end"] - v["start"] for v in old_state["segments"].values())
total_old_words = sum(len(s["text"].split()) for s in old_segments)
AVG_SEC_PER_WORD = total_old_duration / total_old_words
print(f"Mock rate: {AVG_SEC_PER_WORD:.3f} sec/word (derived from real Stage 1 alignment, not invented)\n")

resolved_v2 = []
cumulative_delta = 0.0

for c in classifications:
    if c["kind"] == "unchanged":
        old = old_state["segments"][c["old_id"]]
        if cumulative_delta == 0.0:
            action, reason = "REUSE", "unchanged text, no upstream edits before it"
        else:
            action = "VERIFY_SHIFT"
            reason = f"unchanged text, but shifted by {cumulative_delta:+.2f}s from upstream edits"
        resolved_v2.append({"id": c["new_id"], "text": c["new_text"],
                             "start": old["start"] + cumulative_delta, "end": old["end"] + cumulative_delta,
                             "action": action, "reason": reason})

    elif c["kind"] == "edited":
        old = old_state["segments"][c["old_id"]]
        sim = cosine_similarity(get_embedding(c["old_text"]), get_embedding(c["new_text"]))
        word_delta = len(c["new_text"].split()) - len(c["old_text"].split())
        mock_duration_change = round(word_delta * AVG_SEC_PER_WORD, 2)
        action = "PATCH" if sim >= SIMILARITY_THRESHOLD else "FULL_REALIGN"
        reason = (f"reworded (similarity {sim:.3f}), mock duration change {mock_duration_change:+.2f}s "
                  f"(MOCKED -- pending real windowed re-alignment)")
        new_start = old["start"] + cumulative_delta
        new_end = old["end"] + cumulative_delta + mock_duration_change
        cumulative_delta += mock_duration_change
        resolved_v2.append({"id": c["new_id"], "text": c["new_text"],
                             "start": new_start, "end": new_end, "action": action, "reason": reason})

    elif c["kind"] == "inserted":
        mock_duration = round(len(c["new_text"].split()) * AVG_SEC_PER_WORD, 2)
        prev_end = resolved_v2[-1]["end"] if resolved_v2 else 0.0
        new_start = prev_end + 0.3
        new_end = new_start + mock_duration
        cumulative_delta += (new_end - prev_end)
        resolved_v2.append({"id": c["new_id"], "text": c["new_text"],
                             "start": new_start, "end": new_end, "action": "NEW",
                             "reason": f"no prior timestamp exists, mock duration {mock_duration:.2f}s "
                                       f"(MOCKED -- pending real alignment)"})

    elif c["kind"] == "deleted":
        old = old_state["segments"][c["old_id"]]
        removed_duration = old["end"] - old["start"]
        cumulative_delta -= removed_duration
        resolved_v2.append({"id": c["old_id"], "text": c["old_text"], "start": None, "end": None,
                             "action": "REMOVE",
                             "reason": f"segment deleted, removes {removed_duration:.2f}s from the timeline"})

print(f"Final cumulative delta by end of transcript: {cumulative_delta:+.2f}s\n")

for r in resolved_v2:
    if r["action"] == "REUSE":
        continue
    span = f"[{r['start']:.2f}s - {r['end']:.2f}s]" if r["start"] is not None else ""
    print(f"Decision: {r['action']:<12} {r['id']}  {span}")
    print(f"Reason:   {r['reason']}\n")

In [ ]:
from datetime import datetime, timezone

DECISIONS_LOG_PATH = VIDEO_DIR / "decisions.log"

with open(DECISIONS_LOG_PATH, "a") as f:
    f.write(f"\n{'='*70}\n")
    f.write(f"Run at: {datetime.now(timezone.utc).isoformat()}\n")
    f.write(f"Old transcript: {OLD_TRANSCRIPT_PATH.name}\n")
    f.write(f"New transcript: {NEW_TRANSCRIPT_PATH.name}\n")
    f.write(f"Summary: {summary}\n")
    f.write(f"Final cumulative delta: {cumulative_delta:+.2f}s\n")
    f.write(f"{'='*70}\n\n")

    for r in resolved_v2:
        span = f"[{r['start']:.2f}s - {r['end']:.2f}s]" if r["start"] is not None else ""
        f.write(f"Decision: {r['action']:<12} {r['id']}  {span}\n")
        f.write(f"Reason:   {r['reason']}\n\n")

print(f"Appended {len(resolved_v2)} decision entries to {DECISIONS_LOG_PATH}")

## Summary:
Given an old state and a new transcript, the system now produces, per segment, a specific action (REUSE / PATCH / FULL_REALIGN / NEW / VERIFY_SHIFT / REMOVE) with a plain-language justification and a concrete (mocked, clearly labeled) timestamp — and critically, it correctly compounds multiple changes across the file rather than treating each edit in isolation.

## Phase 4: Clip Reuse
In this final stage we:
1. Decide which of the previously cut clip files are still valid, and which need re-cutting. A clip is only safe to reuse if both its source text is unchanged and its resolved timestamp hasn't moved at all.
2. Either one changing (a reword, or just a downstream time shift from an earlier edit) means the old clip file no longer matches reality and must be re-cut.

In [ ]:
# Reload the manifest from disk (not from an in-memory variable), same
# principle as reloading state.json earlier -- a real second run wouldn't
# have last run's Python variables sitting around.
old_manifest = json.load(open(MANIFEST_PATH))
old_clips_by_segment = {c["segment_id"]: c for c in old_manifest["clips"]}

new_by_id = {s["id"]: s for s in new_segments}
TIMESTAMP_TOLERANCE = 0.05  # seconds -- allows for float rounding, not real drift

clip_decisions = []

for r in resolved_v2:
    if r["action"] == "REMOVE":
        # If this deleted segment used to have a clip, that clip is now orphaned.
        old_clip = old_clips_by_segment.get(r["id"])
        if old_clip:
            clip_decisions.append({"id": r["id"], "action": "RETIRE_CLIP",
                                    "reason": f"source segment deleted -- {old_clip['filename']} no longer has a source"})
        continue

    seg = new_by_id[r["id"]]
    if not seg.get("make_clip"):
        continue  # not a clip candidate, nothing to reuse or cut

    old_clip = old_clips_by_segment.get(r["id"])
    if old_clip is None:
        clip_decisions.append({"id": r["id"], "action": "CUT_NEW",
                                "reason": "newly flagged as a clip -- no prior clip file exists"})
        continue

    start_matches = abs(old_clip["start"] - r["start"]) < TIMESTAMP_TOLERANCE
    end_matches = abs(old_clip["end"] - r["end"]) < TIMESTAMP_TOLERANCE

    if start_matches and end_matches:
        clip_decisions.append({"id": r["id"], "action": "REUSE_CLIP",
                                "reason": f"text and timestamp both unchanged -- {old_clip['filename']} is still valid"})
    else:
        clip_decisions.append({"id": r["id"], "action": "RECUT",
                                "reason": f"timestamp moved ({old_clip['start']:.2f}-{old_clip['end']:.2f}s -> "
                                          f"{r['start']:.2f}-{r['end']:.2f}s) -- {old_clip['filename']} is stale"})

summary = {}
for d in clip_decisions:
    summary[d["action"]] = summary.get(d["action"], 0) + 1
print("Clip decision summary:", summary, "\n")

for d in clip_decisions:
    print(f"Decision: {d['action']:<12} {d['id']}")
    print(f"Reason:   {d['reason']}\n")